# Building and Optimising a RAG Pipeline
### From Naive Retrieval to DSPy-Style Automatic Optimisation

**NLP Portfolio -- Retrieval-Augmented Generation**

Most RAG tutorials show the mechanics: embed, retrieve, generate.
This notebook goes deeper -- we build a RAG system **and then systematically
optimise every component**, borrowing ideas from DSPy's declarative
programming model.

### What You Will Learn

| Section | What | Why |
|---|---|---|
| 1. Foundations | Build a RAG pipeline from scratch | Understand every moving part |
| 2. Failure Modes | Diagnose why naive RAG gives bad answers | Know what to fix |
| 3. Component Profiling | Measure retrieval and generation quality independently | Isolate bottlenecks |
| 4. Manual Tuning | Improve prompts and retrieval by hand | See the ceiling of human effort |
| 5. DSPy-Style Optimisation | Automatically optimise the full pipeline | The declarative advantage |
| 6. Comparison | Side-by-side evaluation of all approaches | Quantify the improvement |

### The Core Insight

A RAG pipeline has **two failure modes**: bad retrieval and bad generation.
Fixing one without measuring the other wastes effort. DSPy-style optimisation
treats the entire pipeline as a program and optimises it end-to-end.

## How RAG Actually Works (Under the Hood)

```
         QUERY: 'What causes gradient vanishing in RNNs?'
           |
           v
    +--------------+
    |   RETRIEVER  |  Find the top-k most relevant passages
    +--------------+  from an indexed knowledge base
           |
           v  [passage_1, passage_2, passage_3]
    +--------------+
    |   READER /   |  Generate an answer grounded in the
    |   GENERATOR  |  retrieved passages
    +--------------+
           |
           v
    ANSWER: 'Gradient vanishing occurs because ...'
```

### What Can Go Wrong?

| Failure Mode | Symptom | Root Cause |
|---|---|---|
| **Retrieval miss** | Answer is wrong or hallucinated | Relevant passages were not retrieved |
| **Retrieval noise** | Answer is diluted or confused | Irrelevant passages were retrieved |
| **Generation miss** | Passages are correct but answer is wrong | LLM ignored or misread the context |
| **Faithfulness** | Answer contradicts the passages | LLM hallucinated despite good retrieval |

The key insight: **you must measure retrieval and generation separately**
before you can improve the pipeline.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, math, time, copy, hashlib, textwrap
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Part 1: Knowledge Base

We build a corpus of 25 technical articles on ML/AI topics, chunked into
~120 passages. This is our RAG knowledge base.

In [ ]:
@dataclass
class Passage:
    id: str = ''
    content: str = ''
    source: str = ''
    category: str = ''
    embedding: Optional[np.ndarray] = None
    score: float = 0.0

    def __post_init__(self):
        if not self.id:
            self.id = hashlib.sha256(self.content.encode()).hexdigest()[:10]


ARTICLES = [
    {'title': 'Gradient Descent Variants', 'cat': 'optimisation', 'text': (
        'Gradient descent minimises a loss function by iterating in the direction '
        'of the negative gradient. Stochastic gradient descent updates on each sample. '
        'Mini-batch SGD uses batches of 32-256 for a noise-speed balance. '
        'Momentum accumulates velocity, dampening oscillations. '
        'Adam combines momentum and adaptive per-parameter learning rates using '
        'first and second moment estimates. The update rule is: '
        'theta = theta - lr * m_hat / (sqrt(v_hat) + eps) where m_hat and v_hat '
        'are bias-corrected moment estimates. Learning rate schedules include '
        'cosine annealing, warmup, step decay, and reduce-on-plateau.'
    )},
    {'title': 'Regularisation Methods', 'cat': 'optimisation', 'text': (
        'Regularisation prevents overfitting. L1 (Lasso) adds the sum of absolute '
        'weights to the loss, encouraging sparsity -- many weights become exactly zero. '
        'L2 (Ridge) adds the sum of squared weights, shrinking all weights. '
        'Elastic Net combines L1 and L2. Dropout randomly zeros neurons during '
        'training at rate p, acting as an implicit ensemble. Batch normalisation '
        'normalises layer activations, reducing internal covariate shift. '
        'Early stopping halts training when validation loss plateaus. '
        'Data augmentation artificially increases diversity.'
    )},
    {'title': 'Transformer Architecture', 'cat': 'architecture', 'text': (
        'Transformers use self-attention to model relationships between all tokens '
        'simultaneously. Multi-head attention projects Q, K, V into h subspaces. '
        'Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V. Positional encodings '
        'inject sequence order. The encoder processes input; the decoder generates '
        'output with cross-attention. Pre-training: masked LM (BERT), causal LM '
        '(GPT), seq2seq (T5). Flash Attention reduces memory from O(N^2) to O(N) '
        'using tiling and online softmax. RoPE and ALiBi provide length '
        'generalisation without learned position embeddings.'
    )},
    {'title': 'Recurrent Neural Networks', 'cat': 'architecture', 'text': (
        'RNNs process sequences by maintaining hidden state: h_t = tanh(W_h h_{t-1} '
        '+ W_x x_t + b). Vanilla RNNs suffer from vanishing gradients: gradients '
        'shrink exponentially over long sequences, preventing learning of long-range '
        'dependencies. LSTM solves this with forget, input, and output gates '
        'controlling a memory cell. The forget gate f_t = sigmoid(W_f [h_{t-1}, x_t]) '
        'decides what to discard. GRU simplifies with reset and update gates. '
        'Bidirectional RNNs process sequences in both directions.'
    )},
    {'title': 'Convolutional Neural Networks', 'cat': 'architecture', 'text': (
        'CNNs extract hierarchical spatial features using learnable filters. '
        'A convolution computes output[i,j] = sum of input[i+m,j+n] * kernel[m,n]. '
        'Pooling reduces spatial dimensions. Architectures: VGG (deep uniform), '
        'ResNet (skip connections prevent gradient degradation in deep networks), '
        'EfficientNet (compound scaling of depth/width/resolution), '
        'MobileNet (depthwise separable convolutions for edge deployment). '
        'Transfer learning uses ImageNet-pretrained features fine-tuned on target data.'
    )},
    {'title': 'Attention Mechanisms', 'cat': 'architecture', 'text': (
        'Attention lets models focus on relevant input parts. Bahdanau (additive) '
        'computes score = v^T tanh(W h_t + U h_s). Luong (multiplicative): '
        'score = h_t^T W h_s. Scaled dot-product: QK^T / sqrt(d_k). '
        'Self-attention relates each position to all others in the same sequence. '
        'Cross-attention relates decoder to encoder positions. Multi-head: '
        'project to h subspaces, attend independently, concatenate. '
        'Sparse patterns (Longformer, BigBird) reduce quadratic cost.'
    )},
    {'title': 'Sentence Embeddings', 'cat': 'nlp', 'text': (
        'Sentence embeddings map sentences to dense fixed-size vectors. '
        'SBERT fine-tunes BERT with siamese networks on NLI/STS tasks, enabling '
        'fast cosine similarity search. SimCSE uses contrastive learning with '
        'in-batch negatives. Pooling strategies: CLS token, mean pooling, '
        'max pooling, weighted mean. ANN search scales with HNSW or IVF-PQ '
        '(FAISS). MTEB benchmark evaluates across 58 tasks. '
        'Instruction-tuned embeddings (E5, GTE) improve zero-shot transfer.'
    )},
    {'title': 'Named Entity Recognition', 'cat': 'nlp', 'text': (
        'NER identifies named entities: PERSON, ORG, GPE, DATE, MONEY. '
        'IOB tagging marks B (begin), I (inside), O (outside). '
        'Approaches: rule-based gazetteers, CRF with hand-crafted features, '
        'BiLSTM-CRF with learned embeddings, fine-tuned BERT-NER. '
        'Evaluation uses entity-level F1 (exact span match). '
        'Few-shot NER with prompt tuning reduces annotation needs.'
    )},
    {'title': 'Text Classification', 'cat': 'nlp', 'text': (
        'Text classification assigns categories to documents. Traditional ML: '
        'TF-IDF features with logistic regression or SVM. Deep learning: '
        'CNN-text (Kim 2014), BiLSTM with attention, fine-tuned BERT. '
        'Multi-label: binary cross-entropy per label. Few-shot: SetFit '
        'trains a sentence-transformer on small labelled sets. Zero-shot: '
        'NLI-based classification treats labels as hypotheses.'
    )},
    {'title': 'Question Answering', 'cat': 'nlp', 'text': (
        'QA systems answer questions from text. Extractive QA predicts '
        'start/end token positions (SQuAD, BiDAF, BERT-QA). Open-domain QA '
        'combines retrieval and reading: DPR (Dense Passage Retrieval) + '
        'FiD (Fusion-in-Decoder). RAG (Retrieval Augmented Generation) '
        'retrieves evidence then generates answers. Multi-hop QA requires '
        'reasoning across multiple documents. Evaluation: EM, F1, ROUGE.'
    )},
    {'title': 'BM25 and Inverted Indexes', 'cat': 'ir', 'text': (
        'An inverted index maps terms to posting lists. BM25 scoring: '
        'sum over query terms of IDF * tf_normalised. Parameters: k1=1.5 '
        'controls TF saturation, b=0.75 controls length normalisation. '
        'IDF(t) = log((N - df + 0.5)/(df + 0.5) + 1). '
        'BM25 is the strongest sparse retrieval baseline. '
        'Compression: variable-byte encoding, PFOR-DELTA.'
    )},
    {'title': 'Dense Retrieval', 'cat': 'ir', 'text': (
        'Dense retrieval encodes queries and documents as vectors, '
        'retrieving by maximum inner product search. DPR uses separate BERT '
        'encoders trained with in-batch negatives. Hard negatives from BM25 '
        'improve training. ColBERT late interaction: sum of max per-token '
        'similarities. SPLADE learns sparse representations via BERT MLM '
        'head. Hybrid approaches combine BM25 + dense scores.'
    )},
    {'title': 'Reranking Models', 'cat': 'ir', 'text': (
        'Cross-encoder rerankers score (query, passage) pairs with a '
        'transformer, achieving high accuracy but O(n) cost. MonoT5 '
        'generates true/false tokens for relevance. Listwise rerankers '
        '(RankGPT) use LLMs to rank multiple documents simultaneously. '
        'MMR (Maximal Marginal Relevance) balances relevance and diversity: '
        'score = lambda*sim(q,d) - (1-lambda)*max_j sim(d, selected_j).'
    )},
    {'title': 'RAG Architectures', 'cat': 'ir', 'text': (
        'Retrieval-Augmented Generation augments LLMs with external knowledge. '
        'Naive RAG: retrieve top-k, concatenate, generate. Advanced RAG: '
        'query rewriting, hybrid retrieval, reranking, iterative retrieval. '
        'Modular RAG: separate retriever, reranker, reader components. '
        'Self-RAG: the LLM decides when to retrieve. CRAG (Corrective RAG): '
        'evaluates retrieval quality and falls back to web search. '
        'Chunking strategies: fixed-size, sentence, semantic splitting.'
    )},
    {'title': 'Evaluation Metrics for IR', 'cat': 'ir', 'text': (
        'Precision@k: fraction of top-k that are relevant. '
        'Recall@k: fraction of relevant docs found in top-k. '
        'MRR: mean of 1/rank of first relevant result. '
        'nDCG@k: normalised discounted cumulative gain for graded relevance. '
        'MAP: mean average precision across queries. '
        'Faithfulness: does the generated answer align with retrieved passages. '
        'Answer relevance: does the answer address the question.'
    )},
    {'title': 'Prompt Engineering', 'cat': 'llm', 'text': (
        'Prompt engineering crafts instructions for LLMs. Zero-shot: direct '
        'question with no examples. Few-shot: include input-output examples '
        'in the prompt. Chain-of-thought (CoT): ask the model to reason '
        'step by step. Tree-of-thought: explore multiple reasoning paths. '
        'Self-consistency: sample multiple CoT paths and majority vote. '
        'System prompts define the model persona and constraints.'
    )},
    {'title': 'DSPy Framework', 'cat': 'llm', 'text': (
        'DSPy replaces hand-written prompts with declarative signatures. '
        'A Signature declares input/output fields (text -> answer). '
        'Modules wrap signatures: Predict (direct), ChainOfThought (with '
        'reasoning), ReAct (with tool calls). Teleprompters (optimisers) '
        'automatically select demonstrations and instructions: '
        'BootstrapFewShot, MIPROv2, COPRO. The metric function measures '
        'program quality. Programs compose like PyTorch nn.Module.'
    )},
    {'title': 'Fine-Tuning LLMs', 'cat': 'llm', 'text': (
        'Fine-tuning adapts a pre-trained LLM to a specific task. '
        'Full fine-tuning updates all parameters (expensive). '
        'LoRA (Low-Rank Adaptation) freezes the base model and trains '
        'low-rank decomposition matrices A and B: W = W0 + A*B. '
        'QLoRA quantises the base model to 4-bit and applies LoRA. '
        'Instruction tuning trains on (instruction, response) pairs. '
        'RLHF uses human preferences to align model behaviour.'
    )},
    {'title': 'Vector Databases', 'cat': 'infra', 'text': (
        'Vector databases store and search high-dimensional embeddings. '
        'HNSW index: hierarchical navigable small world graph for O(log N) '
        'approximate nearest neighbor search. IVF: inverted file index '
        'partitions space into clusters. PQ: product quantisation compresses '
        'vectors. Pinecone, Weaviate, Qdrant, Milvus, Chroma are popular '
        'options. Metadata filtering narrows search before vector similarity.'
    )},
    {'title': 'Chunking Strategies', 'cat': 'infra', 'text': (
        'Chunking splits documents into retrievable units. Fixed-size chunks '
        'of 256-512 tokens with 50-token overlap are simple but may split '
        'concepts. Sentence-based chunking respects sentence boundaries. '
        'Semantic chunking groups sentences by embedding similarity. '
        'Parent-child chunking: small chunks for retrieval, return the '
        'parent paragraph for context. Recursive character splitting '
        'tries paragraph, newline, sentence, then character boundaries.'
    )},
    {'title': 'Ensemble Learning', 'cat': 'ml', 'text': (
        'Bagging trains models on bootstrap samples and aggregates. '
        'Random Forest adds random feature subsets. Boosting trains '
        'sequentially: AdaBoost reweights samples, Gradient Boosting '
        'fits residuals. XGBoost, LightGBM, CatBoost are optimised '
        'implementations. Stacking trains a meta-learner on base model '
        'predictions. Diverse base learners improve ensemble quality.'
    )},
    {'title': 'Cross-Validation', 'cat': 'ml', 'text': (
        'K-fold CV splits data into k folds, trains on k-1, evaluates on '
        'the held-out fold. Stratified k-fold preserves class proportions. '
        'Nested CV separates hyperparameter search from final evaluation '
        'to avoid optimistic bias. Pipeline-aware CV applies preprocessing '
        'within each fold to prevent data leakage. TimeSeriesSplit for '
        'temporal data respects chronological order.'
    )},
    {'title': 'GANs', 'cat': 'generative', 'text': (
        'GANs train Generator G and Discriminator D adversarially. '
        'G tries to fool D; D tries to distinguish real from fake. '
        'Mode collapse: G produces limited diversity. WGAN uses '
        'Wasserstein distance for stable training. StyleGAN2 produces '
        'photorealistic images via progressive growing. CycleGAN does '
        'unpaired image-to-image translation. Evaluation: FID, IS.'
    )},
    {'title': 'Diffusion Models', 'cat': 'generative', 'text': (
        'Diffusion models generate data by learning to reverse a noising '
        'process. Forward process adds Gaussian noise over T steps. '
        'Reverse process (a neural net) denoises step by step. '
        'DDPM (Denoising Diffusion Probabilistic Models) introduced the '
        'framework. Stable Diffusion operates in latent space (VAE encoder '
        'first). Classifier-free guidance balances quality and diversity. '
        'Controlnet adds spatial conditioning (edge maps, depth, pose).'
    )},
    {'title': 'Reinforcement Learning', 'cat': 'rl', 'text': (
        'RL learns policies by maximising cumulative reward through '
        'environment interaction. Q-learning estimates action values. '
        'Policy gradient methods (REINFORCE, PPO) directly optimise the '
        'policy. Actor-critic combines both. DQN uses experience replay '
        'and target networks for stability. PPO clips the policy ratio '
        'for stable updates. Multi-agent RL extends to multiple learners.'
    )},
]

def chunk_article(article, chunk_size=3):
    sents = re.split(r'(?<=[.!?])\s+', article['text'].strip())
    chunks = []
    step = max(1, chunk_size - 1)
    for i in range(0, len(sents), step):
        text = ' '.join(sents[i:i + chunk_size])
        if len(text.strip()) < 30:
            continue
        chunks.append(Passage(
            content=text,
            source=article['title'],
            category=article['cat'],
        ))
    return chunks


KNOWLEDGE_BASE = []
for art in ARTICLES:
    KNOWLEDGE_BASE.extend(chunk_article(art))

print(f'Knowledge base: {len(ARTICLES)} articles -> {len(KNOWLEDGE_BASE)} passages')
cats = Counter(p.category for p in KNOWLEDGE_BASE)
for cat, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f'  {cat:<15s}: {n} passages')

## Embedding Engine

Fitted once, shared by all pipeline variants.

In [ ]:
class Embedder:
    def __init__(self, dim=96):
        self.dim = dim
        self.vocab = {}
        self.idf = {}
        self._proj = None

    def fit(self, texts):
        df = Counter()
        for text in texts:
            for t in set(self._tok(text)):
                df[t] += 1
        self.vocab = {t: i for i, t in enumerate(sorted(df))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + c)) for t, c in df.items()}
        np.random.seed(42)
        self._proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)

    def _tok(self, text):
        return re.findall(r'[a-z0-9]+', text.lower())

    def encode(self, text):
        tf = Counter(self._tok(text))
        v = np.zeros(len(self.vocab))
        for t, cnt in tf.items():
            if t in self.vocab:
                v[self.vocab[t]] = cnt * self.idf.get(t, 1.0)
        proj = v @ self._proj
        norm = np.linalg.norm(proj)
        return proj / norm if norm > 0 else proj


EMB = Embedder(dim=96)
EMB.fit([p.content for p in KNOWLEDGE_BASE])

# Pre-compute all passage embeddings
for p in KNOWLEDGE_BASE:
    p.embedding = EMB.encode(p.content)

print(f'Embedder: vocab={len(EMB.vocab)}, dim={EMB.dim}')

## QA Evaluation Dataset

35 questions with expected answers and source articles (ground truth).
Every pipeline variant is evaluated on this exact same set.

In [ ]:
QA_DATASET = [
    # optimisation
    {'question': 'What is the Adam optimiser update rule?',
     'answer': 'Adam updates parameters using bias-corrected first and second moment estimates: theta = theta - lr * m_hat / (sqrt(v_hat) + eps).',
     'source': 'Gradient Descent Variants', 'type': 'factual'},
    {'question': 'How does momentum improve gradient descent?',
     'answer': 'Momentum accumulates velocity in the gradient direction, dampening oscillations and accelerating convergence.',
     'source': 'Gradient Descent Variants', 'type': 'conceptual'},
    {'question': 'What is the difference between L1 and L2 regularisation?',
     'answer': 'L1 adds the sum of absolute weights encouraging sparsity (many weights become zero), while L2 adds the sum of squared weights, shrinking all weights proportionally.',
     'source': 'Regularisation Methods', 'type': 'comparison'},
    {'question': 'How does dropout prevent overfitting?',
     'answer': 'Dropout randomly zeros neurons during training at rate p, acting as an implicit ensemble of many subnetworks.',
     'source': 'Regularisation Methods', 'type': 'conceptual'},
    # architecture
    {'question': 'What is the scaled dot-product attention formula?',
     'answer': 'Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V.',
     'source': 'Transformer Architecture', 'type': 'factual'},
    {'question': 'Why do vanilla RNNs suffer from vanishing gradients?',
     'answer': 'Gradients shrink exponentially over long sequences because repeated multiplication through the recurrence produces values that approach zero.',
     'source': 'Recurrent Neural Networks', 'type': 'conceptual'},
    {'question': 'How does LSTM solve the vanishing gradient problem?',
     'answer': 'LSTM introduces forget, input, and output gates controlling a memory cell that can maintain information over long sequences without gradient degradation.',
     'source': 'Recurrent Neural Networks', 'type': 'conceptual'},
    {'question': 'What are skip connections in ResNet?',
     'answer': 'Skip connections add the input of a layer directly to its output, preventing gradient degradation in very deep networks.',
     'source': 'Convolutional Neural Networks', 'type': 'factual'},
    {'question': 'How does Flash Attention reduce memory usage?',
     'answer': 'Flash Attention reduces memory from O(N^2) to O(N) by using tiling to process attention blocks and online softmax to avoid materialising the full attention matrix.',
     'source': 'Transformer Architecture', 'type': 'conceptual'},
    {'question': 'What is cross-attention in transformers?',
     'answer': 'Cross-attention relates decoder positions to encoder positions, allowing the decoder to attend to the input representation.',
     'source': 'Attention Mechanisms', 'type': 'factual'},
    # nlp
    {'question': 'How does SBERT create sentence embeddings?',
     'answer': 'SBERT fine-tunes BERT using siamese networks on NLI/STS tasks, producing fixed-size sentence vectors suitable for fast cosine similarity search.',
     'source': 'Sentence Embeddings', 'type': 'conceptual'},
    {'question': 'What is IOB tagging in NER?',
     'answer': 'IOB tagging marks B (begin), I (inside), O (outside) to identify entity span boundaries in sequences.',
     'source': 'Named Entity Recognition', 'type': 'factual'},
    {'question': 'How does SetFit handle few-shot text classification?',
     'answer': 'SetFit trains a sentence-transformer on small labelled sets using contrastive learning, then fits a classification head on the resulting embeddings.',
     'source': 'Text Classification', 'type': 'conceptual'},
    {'question': 'What is DPR and how does it work?',
     'answer': 'Dense Passage Retrieval uses separate BERT encoders for questions and passages, trained with in-batch negatives using dot-product similarity for open-domain QA.',
     'source': 'Question Answering', 'type': 'factual'},
    # ir
    {'question': 'What do the BM25 parameters k1 and b control?',
     'answer': 'k1 (default 1.5) controls term frequency saturation and b (default 0.75) controls document length normalisation.',
     'source': 'BM25 and Inverted Indexes', 'type': 'factual'},
    {'question': 'What is ColBERT late interaction?',
     'answer': 'ColBERT encodes queries and documents at the token level and computes relevance as the sum of maximum per-token similarities between query and passage tokens.',
     'source': 'Dense Retrieval', 'type': 'conceptual'},
    {'question': 'How does a cross-encoder reranker work?',
     'answer': 'A cross-encoder concatenates the query and passage, processes them through a transformer, and outputs a scalar relevance score, achieving high accuracy at O(n) cost.',
     'source': 'Reranking Models', 'type': 'conceptual'},
    {'question': 'What is Self-RAG?',
     'answer': 'In Self-RAG, the LLM itself decides when retrieval is needed, rather than always retrieving for every query.',
     'source': 'RAG Architectures', 'type': 'factual'},
    {'question': 'What is nDCG used to measure?',
     'answer': 'nDCG (normalised discounted cumulative gain) measures ranking quality accounting for graded relevance and position, computed as DCG / IDCG.',
     'source': 'Evaluation Metrics for IR', 'type': 'factual'},
    # llm
    {'question': 'What is chain-of-thought prompting?',
     'answer': 'Chain-of-thought prompting asks the model to reason step by step before producing the final answer, improving performance on complex reasoning tasks.',
     'source': 'Prompt Engineering', 'type': 'conceptual'},
    {'question': 'What are DSPy Signatures?',
     'answer': 'DSPy Signatures declare input and output fields for a transformation (e.g., text -> answer), replacing hand-written prompts with typed declarations.',
     'source': 'DSPy Framework', 'type': 'factual'},
    {'question': 'What is LoRA fine-tuning?',
     'answer': 'LoRA freezes the pre-trained model and trains low-rank decomposition matrices A and B such that W = W0 + A*B, reducing trainable parameters dramatically.',
     'source': 'Fine-Tuning LLMs', 'type': 'factual'},
    {'question': 'How do DSPy Teleprompters work?',
     'answer': 'Teleprompters are optimisers that automatically select demonstrations and instructions by evaluating program variations against a metric function.',
     'source': 'DSPy Framework', 'type': 'conceptual'},
    # infra
    {'question': 'What is HNSW indexing?',
     'answer': 'HNSW (Hierarchical Navigable Small World) builds a multi-layer graph enabling O(log N) approximate nearest neighbor search for high-dimensional vectors.',
     'source': 'Vector Databases', 'type': 'factual'},
    {'question': 'What is semantic chunking?',
     'answer': 'Semantic chunking groups sentences by embedding similarity rather than fixed size, keeping conceptually related sentences together.',
     'source': 'Chunking Strategies', 'type': 'conceptual'},
    {'question': 'What is parent-child chunking?',
     'answer': 'Parent-child chunking uses small chunks for retrieval (better precision) but returns the larger parent paragraph as context (more information for generation).',
     'source': 'Chunking Strategies', 'type': 'conceptual'},
    # ml
    {'question': 'How does gradient boosting differ from bagging?',
     'answer': 'Bagging trains models in parallel on bootstrap samples and aggregates, while gradient boosting trains models sequentially, each fitting the residuals of the previous one.',
     'source': 'Ensemble Learning', 'type': 'comparison'},
    {'question': 'What is nested cross-validation?',
     'answer': 'Nested CV uses an inner loop for hyperparameter search and an outer loop for final evaluation, avoiding optimistic bias from using the same data for both.',
     'source': 'Cross-Validation', 'type': 'conceptual'},
    # generative
    {'question': 'What causes mode collapse in GANs?',
     'answer': 'Mode collapse occurs when the generator produces only a limited variety of outputs, failing to capture the full diversity of the data distribution.',
     'source': 'GANs', 'type': 'conceptual'},
    {'question': 'How do diffusion models generate images?',
     'answer': 'Diffusion models learn to reverse a forward noising process, starting from pure noise and iteratively denoising step by step to produce clean data.',
     'source': 'Diffusion Models', 'type': 'conceptual'},
    {'question': 'What is classifier-free guidance?',
     'answer': 'Classifier-free guidance balances quality and diversity by interpolating between conditional and unconditional diffusion model predictions.',
     'source': 'Diffusion Models', 'type': 'conceptual'},
    # rl
    {'question': 'How does PPO stabilise policy updates?',
     'answer': 'PPO clips the policy ratio to a small range (typically 0.8-1.2), preventing destructively large policy updates.',
     'source': 'Reinforcement Learning', 'type': 'factual'},
    # cross-domain
    {'question': 'How is RLHF used to align language models?',
     'answer': 'RLHF (Reinforcement Learning from Human Feedback) trains a reward model on human preferences and then uses PPO to fine-tune the LLM to maximise that reward.',
     'source': 'Fine-Tuning LLMs', 'type': 'conceptual'},
    {'question': 'What is the difference between RAG and fine-tuning?',
     'answer': 'RAG retrieves external knowledge at inference time without changing model weights, while fine-tuning updates model parameters on task-specific data during training.',
     'source': 'RAG Architectures', 'type': 'comparison'},
    {'question': 'How does MIPROv2 optimise DSPy programs?',
     'answer': 'MIPROv2 jointly optimises prompt instructions and demonstration selection using multi-prompt instruction and demonstration proposal and evaluation.',
     'source': 'DSPy Framework', 'type': 'factual'},
]

# Build ground truth: question -> set of relevant passage IDs
source_to_ids = defaultdict(set)
for p in KNOWLEDGE_BASE:
    source_to_ids[p.source].add(p.id)

for q in QA_DATASET:
    q['relevant_ids'] = source_to_ids.get(q['source'], set())

print(f'QA dataset: {len(QA_DATASET)} questions')
print(f'Types: {dict(Counter(q["type"] for q in QA_DATASET))}')

## Evaluation Metrics

We measure **retrieval quality** and **answer quality** separately --
this is essential for diagnosing failures.

| Metric | Measures | Target |
|---|---|---|
| Recall@k | Did we retrieve the relevant passages? | Retriever |
| MRR | Where did the first relevant passage rank? | Retriever |
| Answer F1 | Token overlap between generated and expected answer | Generator |
| Faithfulness | Is the answer grounded in retrieved passages? | Generator |
| End-to-end accuracy | Is the full pipeline answer correct? | Pipeline |

In [ ]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    if not relevant_ids: return 0.0
    return len(set(retrieved_ids[:k]) & relevant_ids) / len(relevant_ids)

def mrr(retrieved_ids, relevant_ids):
    for rank, pid in enumerate(retrieved_ids, 1):
        if pid in relevant_ids:
            return 1.0 / rank
    return 0.0

def token_f1(prediction, reference):
    pred_tok = set(re.findall(r'[a-z0-9]+', prediction.lower()))
    ref_tok = set(re.findall(r'[a-z0-9]+', reference.lower()))
    if not pred_tok or not ref_tok: return 0.0
    common = pred_tok & ref_tok
    if not common: return 0.0
    p = len(common) / len(pred_tok)
    r = len(common) / len(ref_tok)
    return 2 * p * r / (p + r)

def faithfulness(answer, passages):
    # What fraction of answer tokens appear in the retrieved passages?
    ans_tok = set(re.findall(r'[a-z0-9]+', answer.lower()))
    ctx_tok = set()
    for p in passages:
        ctx_tok.update(re.findall(r'[a-z0-9]+', p.lower()))
    if not ans_tok: return 0.0
    return len(ans_tok & ctx_tok) / len(ans_tok)

print('Metrics: Recall@k, MRR, Token F1, Faithfulness')

## Part 2: Building the RAG Pipeline

### The Retriever

We start with a simple BM25 retriever and a dense retriever,
then combine them with hybrid fusion.

In [ ]:
class BM25Retriever:
    name = 'BM25'

    def __init__(self, passages, k1=1.5, b=0.75):
        self.passages = passages
        self.k1, self.b = k1, b
        self._tok_map = {}
        self._df = Counter()
        for p in passages:
            toks = re.findall(r'[a-z0-9]+', p.content.lower())
            self._tok_map[p.id] = toks
            for t in set(toks):
                self._df[t] += 1
        self._avgdl = float(np.mean([len(t) for t in self._tok_map.values()]))
        self._n = len(passages)

    def retrieve(self, query, top_k=5):
        qtoks = re.findall(r'[a-z0-9]+', query.lower())
        scores = {}
        id_map = {p.id: p for p in self.passages}
        for p in self.passages:
            toks = self._tok_map[p.id]
            tf = Counter(toks)
            dl = len(toks)
            s = 0.0
            for qt in qtoks:
                f = tf.get(qt, 0)
                df = self._df.get(qt, 0)
                idf = math.log((self._n - df + 0.5) / (df + 0.5) + 1)
                tfn = f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * dl / self._avgdl))
                s += idf * tfn
            if s > 0:
                scores[p.id] = s
        ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
        results = []
        for pid, sc in ranked:
            p2 = copy.copy(id_map[pid])
            p2.score = round(sc, 5)
            results.append(p2)
        return results


class DenseRetriever:
    name = 'Dense'

    def __init__(self, passages, embedder):
        self.passages = passages
        self.embedder = embedder
        self._mat = np.stack([p.embedding for p in passages])

    def retrieve(self, query, top_k=5):
        qe = self.embedder.encode(query)
        sims = self._mat @ qe
        topk = np.argsort(sims)[::-1][:top_k]
        results = []
        for idx in topk:
            p2 = copy.copy(self.passages[idx])
            p2.score = round(float(sims[idx]), 5)
            results.append(p2)
        return results


class HybridRetriever:
    name = 'Hybrid RRF'

    def __init__(self, bm25, dense, k_rrf=60):
        self.bm25 = bm25
        self.dense = dense
        self.k_rrf = k_rrf

    def retrieve(self, query, top_k=5):
        bm25_r = self.bm25.retrieve(query, top_k=top_k * 3)
        dense_r = self.dense.retrieve(query, top_k=top_k * 3)
        rrf = {}
        doc_map = {}
        for rank, p in enumerate(bm25_r):
            rrf[p.id] = rrf.get(p.id, 0) + 1.0 / (self.k_rrf + rank + 1)
            doc_map[p.id] = p
        for rank, p in enumerate(dense_r):
            rrf[p.id] = rrf.get(p.id, 0) + 1.0 / (self.k_rrf + rank + 1)
            if p.id not in doc_map:
                doc_map[p.id] = p
        ranked = sorted(rrf.items(), key=lambda x: -x[1])[:top_k]
        return [copy.copy(doc_map[pid]) for pid, _ in ranked]


bm25_ret = BM25Retriever(KNOWLEDGE_BASE)
dense_ret = DenseRetriever(KNOWLEDGE_BASE, EMB)
hybrid_ret = HybridRetriever(bm25_ret, dense_ret)
print('Retrievers ready: BM25, Dense, Hybrid RRF')

### The Answer Generator

Simulates an LLM that reads retrieved passages and produces answers.
Its accuracy depends on:
1. **Passage quality**: relevant passages -> better answers
2. **Prompt quality**: better instructions -> more faithful extraction
3. **Chain-of-thought**: reasoning before answering -> fewer hallucinations
4. **Demonstrations**: few-shot examples -> consistent format

In production, replace with `dspy.ChainOfThought('context, question -> answer')`.

In [ ]:
class AnswerGenerator:
    STOPWORDS = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
                 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
                 'would', 'could', 'should', 'may', 'might', 'can', 'shall',
                 'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from',
                 'as', 'into', 'about', 'between', 'through', 'during',
                 'and', 'but', 'or', 'nor', 'not', 'so', 'yet', 'if', 'then',
                 'it', 'its', 'that', 'this', 'than', 'what', 'which', 'how'}

    def __init__(self, noise=0.10):
        self.noise = noise

    def generate(self, question, passages, prompt_quality=0.3,
                 chain_of_thought=False, demos=None):
        if not passages:
            return 'I cannot answer without context.'

        # Extract key tokens from the question
        q_tokens = set(re.findall(r'[a-z0-9]+', question.lower())) - self.STOPWORDS

        # Score each passage for relevance to the question
        scored_passages = []
        for p in passages:
            p_tokens = set(re.findall(r'[a-z0-9]+', p.content.lower()))
            overlap = len(q_tokens & p_tokens)
            scored_passages.append((overlap, p))
        scored_passages.sort(key=lambda x: -x[0])

        # Pick the best passage and extract a relevant snippet
        best_passage = scored_passages[0][1]
        sentences = re.split(r'(?<=[.!?])\s+', best_passage.content)

        # Score sentences
        sent_scores = []
        for sent in sentences:
            s_tokens = set(re.findall(r'[a-z0-9]+', sent.lower()))
            score = len(q_tokens & s_tokens)
            # Boost for question type keywords in sentence
            if any(w in sent.lower() for w in ['because', 'causes', 'prevents',
                                                'enables', 'allows', 'combines']):
                score += 1.5
            sent_scores.append((score, sent))
        sent_scores.sort(key=lambda x: -x[0])

        # Take top 1-3 sentences based on prompt quality
        n_sents = 1 + int(prompt_quality * 2)  # better prompt -> more context used
        answer_parts = [s for _, s in sent_scores[:n_sents]]

        # Chain-of-thought improvement: also consider other passages
        if chain_of_thought and len(scored_passages) > 1:
            second_best = scored_passages[1][1]
            second_sents = re.split(r'(?<=[.!?])\s+', second_best.content)
            for sent in second_sents:
                s_tokens = set(re.findall(r'[a-z0-9]+', sent.lower()))
                if len(q_tokens & s_tokens) >= 2:
                    answer_parts.append(sent)
                    break

        # Demonstration bonus: use examples to format answer
        if demos and len(demos) >= 2:
            # Having demos guides concise structured answers
            answer_parts = answer_parts[:2]  # trim to concise like demos

        # Add noise proportional to passage count / prompt quality
        if np.random.random() < self.noise * (1 - prompt_quality):
            answer_parts.append('This involves complex trade-offs.')

        return ' '.join(answer_parts)


gen = AnswerGenerator(noise=0.10)
print('AnswerGenerator ready.')

## Part 3: Naive RAG Pipeline

The simplest possible setup: retrieve top-3, concatenate, generate.
No prompt tuning, no reranking, no reasoning.

In [ ]:
class RAGPipeline:
    def __init__(self, retriever, generator, top_k=3,
                 prompt_quality=0.3, chain_of_thought=False, demos=None,
                 name='RAG'):
        self.retriever = retriever
        self.generator = generator
        self.top_k = top_k
        self.prompt_quality = prompt_quality
        self.cot = chain_of_thought
        self.demos = demos
        self.name = name

    def answer(self, question):
        passages = self.retriever.retrieve(question, top_k=self.top_k)
        answer = self.generator.generate(
            question, passages,
            prompt_quality=self.prompt_quality,
            chain_of_thought=self.cot,
            demos=self.demos,
        )
        return answer, passages


def evaluate_pipeline(pipeline, qa_data, label=''):
    results = []
    for q in qa_data:
        answer, passages = pipeline.answer(q['question'])
        retrieved_ids = [p.id for p in passages]
        passage_texts = [p.content for p in passages]

        rec = recall_at_k(retrieved_ids, q['relevant_ids'], k=pipeline.top_k)
        m = mrr(retrieved_ids, q['relevant_ids'])
        f1 = token_f1(answer, q['answer'])
        faith = faithfulness(answer, passage_texts)

        results.append({
            'pipeline': label or pipeline.name,
            'question': q['question'][:60],
            'type': q['type'],
            'recall': rec,
            'mrr': m,
            'answer_f1': f1,
            'faithfulness': faith,
            'answer_preview': answer[:80],
        })
    return pd.DataFrame(results)


# Naive RAG: BM25, no prompt tuning, no CoT
naive_pipe = RAGPipeline(
    retriever=bm25_ret, generator=gen,
    top_k=3, prompt_quality=0.3,
    chain_of_thought=False, name='Naive RAG'
)

naive_df = evaluate_pipeline(naive_pipe, QA_DATASET)
naive_summary = naive_df[['recall', 'mrr', 'answer_f1', 'faithfulness']].mean()

print('NAIVE RAG RESULTS')
print('=' * 50)
for m, v in naive_summary.items():
    print(f'  {m:<15s}: {v:.4f}')

## Part 3: Diagnosing Failures

Before optimising, we need to understand **where** the pipeline fails.
Is it a retrieval problem or a generation problem?

This diagnostic step is crucial -- most RAG tutorials skip it, leading
practitioners to optimise the wrong component.

In [ ]:
# Classify each failure
failure_types = {'retrieval_miss': 0, 'generation_miss': 0,
                 'both_miss': 0, 'success': 0}

diagnosis_rows = []

for _, row in naive_df.iterrows():
    ret_ok = row['recall'] > 0
    gen_ok = row['answer_f1'] > 0.3

    if ret_ok and gen_ok:
        failure_types['success'] += 1
        dx = 'success'
    elif not ret_ok:
        if gen_ok:
            failure_types['success'] += 1  # lucky generation despite bad retrieval
            dx = 'lucky_gen'
        else:
            failure_types['retrieval_miss'] += 1
            dx = 'retrieval_miss'
    else:
        failure_types['generation_miss'] += 1
        dx = 'generation_miss'

    diagnosis_rows.append({'question': row['question'], 'diagnosis': dx,
                           'recall': row['recall'], 'answer_f1': row['answer_f1']})

dx_df = pd.DataFrame(diagnosis_rows)

print('FAILURE DIAGNOSIS')
print('=' * 50)
for ftype, count in failure_types.items():
    pct = count / len(naive_df) * 100
    bar = '#' * int(pct / 2)
    print(f'  {ftype:<18s}: {count:>3} ({pct:5.1f}%) {bar}')

print('\nSample retrieval misses:')
for _, row in dx_df[dx_df['diagnosis'] == 'retrieval_miss'].head(3).iterrows():
    print(f'  Q: {row["question"]}')
    print(f'     Recall: {row["recall"]:.2f} | F1: {row["answer_f1"]:.3f}')

In [ ]:
# Visualise the diagnosis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: retrieval quality vs generation quality
colors = {'success': '#2ecc71', 'retrieval_miss': '#e74c3c',
          'generation_miss': '#f39c12', 'lucky_gen': '#3498db'}
for dx, grp in dx_df.groupby('diagnosis'):
    axes[0].scatter(grp['recall'], grp['answer_f1'],
                    c=colors.get(dx, '#95a5a6'), label=dx, s=60, alpha=0.7)
axes[0].set_xlabel('Retrieval Recall')
axes[0].set_ylabel('Answer F1')
axes[0].set_title('Retrieval Quality vs Generation Quality')
axes[0].legend(fontsize=8)
axes[0].axhline(0.3, color='#bdc3c7', linestyle='--', alpha=0.5)
axes[0].axvline(0.01, color='#bdc3c7', linestyle='--', alpha=0.5)

# Pie: failure distribution
vals = [v for v in failure_types.values() if v > 0]
labs = [k for k, v in failure_types.items() if v > 0]
pie_colors = [colors.get(k, '#95a5a6') for k in labs]
axes[1].pie(vals, labels=labs, autopct='%1.0f%%', colors=pie_colors,
            startangle=90)
axes[1].set_title('Failure Type Distribution')

plt.tight_layout()
plt.show()

## Part 4: Manual Tuning

Now we improve the pipeline by hand -- the traditional approach:

1. **Better retriever**: switch to hybrid (BM25 + Dense with RRF)
2. **Better prompt**: raise prompt quality (simulate careful prompt engineering)
3. **Retrieve more**: top-5 instead of top-3

This represents hours of manual iteration.

In [ ]:
# Manually tuned RAG
manual_pipe = RAGPipeline(
    retriever=hybrid_ret, generator=gen,
    top_k=5, prompt_quality=0.55,  # hand-tuned prompt
    chain_of_thought=False, name='Manual-Tuned RAG'
)

manual_df = evaluate_pipeline(manual_pipe, QA_DATASET)
manual_summary = manual_df[['recall', 'mrr', 'answer_f1', 'faithfulness']].mean()

print('MANUAL-TUNED RAG RESULTS')
print('=' * 50)
for m, v in manual_summary.items():
    print(f'  {m:<15s}: {v:.4f}')

print('\nImprovement over naive:')
for m in ['recall', 'mrr', 'answer_f1', 'faithfulness']:
    delta = manual_summary[m] - naive_summary[m]
    print(f'  {m:<15s}: {"+" if delta >= 0 else ""}{delta:.4f}')

## Part 5: DSPy-Style Automatic Optimisation

Instead of manually tuning each component, we take the DSPy approach:

1. **Declare** what we want: question + context -> answer
2. **Define a metric**: token F1 against expected answers
3. **Let the optimiser search** for the best configuration

### What the Optimiser Searches Over

| Component | Search Space |
|---|---|
| Retriever | BM25, Dense, Hybrid |
| top_k | 3, 5, 7 |
| Prompt quality | 0.3 to 0.8 (simulates instruction optimisation) |
| Chain-of-thought | On / Off |
| Demonstrations | Selected from training examples |

```python
# Real DSPy code for this pipeline:
class AnswerQuestion(dspy.Signature):
    context: list[str] = dspy.InputField(desc='retrieved passages')
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc='factual answer from context')

class RAG(dspy.Module):
    def __init__(self, k=5):
        self.retrieve = dspy.Retrieve(k=k)
        self.answer = dspy.ChainOfThought(AnswerQuestion)

    def forward(self, question):
        context = self.retrieve(question).passages
        return self.answer(context=context, question=question)

# Optimise end-to-end
optimizer = dspy.MIPROv2(metric=token_f1_metric, auto='medium')
optimised_rag = optimizer.compile(RAG(), trainset=train, valset=val)
```

In [ ]:
class DSPyStyleOptimiser:
    def __init__(self, retrievers, generator, train_qs, val_qs):
        self.retrievers = retrievers
        self.generator = generator
        self.train_qs = train_qs
        self.val_qs = val_qs

    def _eval_config(self, retriever, top_k, prompt_q, cot, demos, qs):
        scores = []
        for q in qs:
            passages = retriever.retrieve(q['question'], top_k=top_k)
            answer = self.generator.generate(
                q['question'], passages,
                prompt_quality=prompt_q, chain_of_thought=cot, demos=demos
            )
            scores.append(token_f1(answer, q['answer']))
        return float(np.mean(scores))

    def _select_demos(self, retriever, train_qs):
        # BootstrapFewShot: find examples where the pipeline succeeds
        successful = []
        for q in train_qs:
            passages = retriever.retrieve(q['question'], top_k=5)
            answer = self.generator.generate(
                q['question'], passages, prompt_quality=0.5, chain_of_thought=True
            )
            f1 = token_f1(answer, q['answer'])
            if f1 > 0.35:
                successful.append({'question': q['question'],
                                  'answer': answer, 'f1': f1})
        # Pick diverse demos
        successful.sort(key=lambda x: -x['f1'])
        return successful[:5]

    def optimise(self):
        print('DSPy-style optimisation starting...')
        print(f'  Train: {len(self.train_qs)} | Val: {len(self.val_qs)}')

        best_score = 0
        best_config = {}
        trials = []

        for ret in self.retrievers:
            # Phase 1: Bootstrap demonstrations
            demos = self._select_demos(ret, self.train_qs)
            print(f'  [{ret.name}] Found {len(demos)} good demonstrations')

            # Phase 2: Search over hyperparameters
            for top_k in [3, 5, 7]:
                for prompt_q in [0.3, 0.5, 0.7]:
                    for cot in [False, True]:
                        for use_demos in [False, True]:
                            d = demos if use_demos else None
                            score = self._eval_config(
                                ret, top_k, prompt_q, cot, d, self.val_qs
                            )
                            trial = {
                                'retriever': ret.name,
                                'top_k': top_k,
                                'prompt_quality': prompt_q,
                                'chain_of_thought': cot,
                                'demos': use_demos,
                                'val_f1': round(score, 4),
                            }
                            trials.append(trial)
                            if score > best_score:
                                best_score = score
                                best_config = trial.copy()
                                best_config['demo_data'] = d

        print(f'\n  Evaluated {len(trials)} configurations')
        print(f'  Best val F1: {best_score:.4f}')
        print(f'  Best config: {best_config["retriever"]}, '
              f'top_k={best_config["top_k"]}, '
              f'prompt_q={best_config["prompt_quality"]}, '
              f'CoT={best_config["chain_of_thought"]}, '
              f'demos={best_config["demos"]}')

        return best_config, pd.DataFrame(trials)


# Split QA data: 60% train, 40% val (test is the full set for final eval)
split = int(len(QA_DATASET) * 0.6)
train_qs = QA_DATASET[:split]
val_qs = QA_DATASET[split:]

optimiser = DSPyStyleOptimiser(
    retrievers=[bm25_ret, dense_ret, hybrid_ret],
    generator=gen,
    train_qs=train_qs,
    val_qs=val_qs,
)

best_config, trial_df = optimiser.optimise()

### Optimisation Landscape

Understanding *what* the optimiser searched reveals insights about the
pipeline that manual tuning would miss.

In [ ]:
# What the optimiser explored
print(f'Total trials: {len(trial_df)}')
print(f'Score range: {trial_df["val_f1"].min():.4f} to {trial_df["val_f1"].max():.4f}')
print(f'\nTop 10 configurations:')
top10 = trial_df.nlargest(10, 'val_f1')
print(top10.to_string(index=False))

# Aggregate insights
print('\nAverage F1 by retriever:')
for name, grp in trial_df.groupby('retriever'):
    print(f'  {name:<12s}: {grp["val_f1"].mean():.4f} (best: {grp["val_f1"].max():.4f})')

print('\nChain-of-thought impact:')
for cot, grp in trial_df.groupby('chain_of_thought'):
    print(f'  CoT={str(cot):<6s}: {grp["val_f1"].mean():.4f}')

print('\nDemonstration impact:')
for demos, grp in trial_df.groupby('demos'):
    print(f'  demos={str(demos):<6s}: {grp["val_f1"].mean():.4f}')

In [ ]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['By Retriever', 'By top_k', 'CoT vs No-CoT'])

# By retriever
for ret_name in ['BM25', 'Dense', 'Hybrid RRF']:
    grp = trial_df[trial_df['retriever'] == ret_name]
    fig.add_trace(go.Box(y=grp['val_f1'], name=ret_name), row=1, col=1)

# By top_k
for k in [3, 5, 7]:
    grp = trial_df[trial_df['top_k'] == k]
    fig.add_trace(go.Box(y=grp['val_f1'], name=f'k={k}',
                         showlegend=False), row=1, col=2)

# CoT
for cot in [False, True]:
    grp = trial_df[trial_df['chain_of_thought'] == cot]
    fig.add_trace(go.Box(y=grp['val_f1'],
                         name=f'CoT={cot}', showlegend=False),
                  row=1, col=3)

fig.update_layout(height=400, template='plotly_white',
                  title_text='Optimisation Landscape: What Matters Most?')
fig.update_yaxes(title_text='Val F1', range=[0, 1])
fig.show()

### Build the Optimised Pipeline

In [ ]:
# Build the optimised pipeline using the best configuration
ret_map = {'BM25': bm25_ret, 'Dense': dense_ret, 'Hybrid RRF': hybrid_ret}
best_ret = ret_map[best_config['retriever']]

opt_pipe = RAGPipeline(
    retriever=best_ret, generator=gen,
    top_k=best_config['top_k'],
    prompt_quality=best_config['prompt_quality'],
    chain_of_thought=best_config['chain_of_thought'],
    demos=best_config.get('demo_data'),
    name='DSPy-Optimised RAG',
)

opt_df = evaluate_pipeline(opt_pipe, QA_DATASET)
opt_summary = opt_df[['recall', 'mrr', 'answer_f1', 'faithfulness']].mean()

print('DSPY-OPTIMISED RAG RESULTS')
print('=' * 50)
for m, v in opt_summary.items():
    print(f'  {m:<15s}: {v:.4f}')

In [ ]:
# Also build a Chain-of-Thought only pipeline (no full optimisation)
cot_pipe = RAGPipeline(
    retriever=hybrid_ret, generator=gen,
    top_k=5, prompt_quality=0.45,
    chain_of_thought=True, name='RAG + CoT'
)

cot_df = evaluate_pipeline(cot_pipe, QA_DATASET)
cot_summary = cot_df[['recall', 'mrr', 'answer_f1', 'faithfulness']].mean()

print('RAG + CoT RESULTS')
print('=' * 50)
for m, v in cot_summary.items():
    print(f'  {m:<15s}: {v:.4f}')

## Part 6: Full Comparison

All pipeline variants evaluated side-by-side.

In [ ]:
ALL_PIPELINES = {
    'Naive RAG': naive_summary,
    'Manual-Tuned': manual_summary,
    'RAG + CoT': cot_summary,
    'DSPy-Optimised': opt_summary,
}

comp_rows = []
for name, s in ALL_PIPELINES.items():
    comp_rows.append({
        'Pipeline': name,
        'Recall': round(s['recall'], 4),
        'MRR': round(s['mrr'], 4),
        'Answer F1': round(s['answer_f1'], 4),
        'Faithfulness': round(s['faithfulness'], 4),
    })

comp_df = pd.DataFrame(comp_rows)
print('PIPELINE COMPARISON')
print('=' * 65)
print(comp_df.to_string(index=False))

# Lift over naive
print('\nLift over Naive RAG:')
for name, s in ALL_PIPELINES.items():
    if name != 'Naive RAG':
        lift = s['answer_f1'] - naive_summary['answer_f1']
        print(f'  {name:<18s}: {"+" if lift >= 0 else ""}{lift:.4f} answer F1')

### Visualisations

In [ ]:
PIPE_COLORS = {
    'Naive RAG': '#e74c3c',
    'Manual-Tuned': '#f39c12',
    'RAG + CoT': '#3498db',
    'DSPy-Optimised': '#2ecc71',
}

metrics_plot = ['Recall', 'MRR', 'Answer F1', 'Faithfulness']
fig = make_subplots(rows=1, cols=4,
                    subplot_titles=metrics_plot)

for col, metric in enumerate(metrics_plot, 1):
    for _, row in comp_df.iterrows():
        fig.add_trace(go.Bar(
            x=[row['Pipeline']], y=[row[metric]],
            marker_color=PIPE_COLORS.get(row['Pipeline'], '#95a5a6'),
            name=row['Pipeline'], showlegend=(col == 1),
        ), row=1, col=col)

fig.update_layout(height=430, template='plotly_white',
                  title_text='RAG Pipeline Comparison', showlegend=False)
fig.update_yaxes(range=[0, 1])
fig.show()

In [ ]:
trajectory = [
    ('Naive\nRAG', naive_summary['answer_f1'], PIPE_COLORS['Naive RAG']),
    ('Manual\nTuned', manual_summary['answer_f1'], PIPE_COLORS['Manual-Tuned']),
    ('RAG +\nCoT', cot_summary['answer_f1'], PIPE_COLORS['RAG + CoT']),
    ('DSPy\nOptimised', opt_summary['answer_f1'], PIPE_COLORS['DSPy-Optimised']),
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(trajectory))),
    y=[t[1] for t in trajectory],
    mode='lines', line=dict(color='#bdc3c7', width=2, dash='dot'),
    showlegend=False,
))
for i, (label, val, color) in enumerate(trajectory):
    fig.add_trace(go.Scatter(
        x=[i], y=[val], mode='markers+text',
        marker=dict(size=18, color=color, line=dict(color='white', width=2)),
        text=[f'{val:.1%}'], textposition='top center',
        showlegend=False,
    ))

fig.update_layout(
    title='Answer F1 Trajectory: From Naive to Optimised',
    xaxis=dict(tickmode='array', tickvals=list(range(len(trajectory))),
               ticktext=[t[0] for t in trajectory]),
    yaxis_title='Answer F1', yaxis_range=[0, 1],
    template='plotly_white', height=400,
)
fig.show()

In [ ]:
radar_dims = ['Recall', 'MRR', 'Answer F1', 'Faithfulness']

fig = go.Figure()
for name, s in ALL_PIPELINES.items():
    vals = [s['recall'], s['mrr'], s['answer_f1'], s['faithfulness']]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_dims + [radar_dims[0]],
        name=name, fill='toself',
        line_color=PIPE_COLORS[name],
        opacity=0.6,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 1], visible=True)),
    title='Pipeline Quality Radar',
    template='plotly_white', height=480,
)
fig.show()

### Per-Question Deep Dive

In [ ]:
# Merge per-question results
all_dfs = {
    'Naive RAG': naive_df, 'Manual-Tuned': manual_df,
    'RAG + CoT': cot_df, 'DSPy-Optimised': opt_df,
}

per_q = []
for name, df in all_dfs.items():
    for _, row in df.iterrows():
        per_q.append({'pipeline': name, 'question': row['question'],
                      'type': row['type'], 'answer_f1': row['answer_f1']})

per_q_df = pd.DataFrame(per_q)

fig = px.box(per_q_df, x='pipeline', y='answer_f1', color='pipeline',
             color_discrete_map=PIPE_COLORS,
             title='Answer F1 Distribution by Pipeline',
             template='plotly_white',
             labels={'answer_f1': 'Answer F1', 'pipeline': ''})
fig.update_layout(height=420, showlegend=False)
fig.show()

In [ ]:
# By question type
type_summary = per_q_df.groupby(['pipeline', 'type'])['answer_f1'].mean().unstack('pipeline')

fig, ax = plt.subplots(figsize=(12, 5))
type_summary.plot(kind='bar', ax=ax, color=[PIPE_COLORS.get(c, '#95a5a6')
                                            for c in type_summary.columns],
                  rot=0, edgecolor='white', linewidth=0.5)
ax.set_title('Answer F1 by Question Type')
ax.set_ylabel('Mean Answer F1')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Top-k vs prompt quality heatmap from optimisation trials
hm_data = trial_df.groupby(['top_k', 'prompt_quality'])['val_f1'].mean().unstack()

fig = px.imshow(
    hm_data.values,
    x=[str(c) for c in hm_data.columns],
    y=[str(r) for r in hm_data.index],
    color_continuous_scale='YlOrRd',
    text_auto='.3f',
    title='Heatmap: top_k vs Prompt Quality (mean val F1)',
    labels=dict(x='Prompt Quality', y='top_k', color='Val F1'),
)
fig.update_layout(template='plotly_white', height=340)
fig.show()

## Key Lessons

### Lesson 1: Diagnose Before You Optimise

```
Bad approach:  'RAG answers are wrong' -> 'improve the prompt'
Good approach: 'RAG answers are wrong' -> measure retrieval recall
                                        -> if low: fix retriever first
                                        -> if high: fix generator
```

### Lesson 2: Components Interact

Improving retrieval changes which passages the generator sees,
which can make a previously-good prompt worse. This is why
end-to-end optimisation (DSPy-style) outperforms component-by-component tuning.

### Lesson 3: Chain-of-Thought Helps Most on Hard Questions

CoT adds cost (more tokens) but provides the biggest lift on:
- Comparison questions (requires synthesising multiple passages)
- Conceptual questions (requires reasoning, not just extraction)

For simple factual extraction, CoT may not justify the extra cost.

### Lesson 4: Demonstrations Are Surprisingly Powerful

Showing the model 3-5 examples of good answers improves both
format consistency and factual accuracy. DSPy's BootstrapFewShot
automatically selects the demonstrations that maximise your metric.

## Production DSPy RAG Pipeline

### Complete Working Code

```python
import dspy
from dspy.retrieve import ChromadbRM

# 1. Configure LLM and retriever
lm = dspy.LM('ollama_chat/qwen3', api_base='http://localhost:11434')
rm = ChromadbRM(collection_name='docs', persist_directory='./chroma_db', k=5)
dspy.configure(lm=lm, rm=rm)

# 2. Declare signatures
class AnswerQuestion(dspy.Signature):
    context: list[str] = dspy.InputField(desc='retrieved passages')
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc='concise factual answer grounded in context')

# 3. Build pipeline
class RAG(dspy.Module):
    def __init__(self, k=5):
        self.retrieve = dspy.Retrieve(k=k)
        self.answer = dspy.ChainOfThought(AnswerQuestion)

    def forward(self, question):
        context = self.retrieve(question).passages
        return self.answer(context=context, question=question)

# 4. Define metric
def answer_quality(example, pred, trace=None):
    return dspy.evaluate.answer_exact_match(example, pred)

# 5. Optimise
optimizer = dspy.MIPROv2(metric=answer_quality, auto='medium')
optimised = optimizer.compile(RAG(), trainset=train, valset=val)

# 6. Save and deploy
optimised.save('rag_v1.json')
```

### Adding Assertions for Reliability

```python
class ReliableRAG(dspy.Module):
    def __init__(self):
        self.retrieve = dspy.Retrieve(k=5)
        self.answer = dspy.ChainOfThought(AnswerQuestion)

    def forward(self, question):
        context = self.retrieve(question).passages
        result = self.answer(context=context, question=question)

        # Assertion: answer must be grounded in retrieved text
        dspy.Assert(
            any(word in ' '.join(context).lower()
                for word in result.answer.lower().split()[:5]),
            'Answer must be grounded in retrieved passages'
        )
        return result
```

When assertions fail, DSPy automatically retries with feedback.

## Summary

### The RAG Optimisation Journey

```
Naive RAG              Manual Tuning           DSPy-Style Optimisation
-----------            ---------------         ------------------------
BM25 retriever    ->   Hybrid retriever   ->   Searched 3 retrievers
No prompt tuning  ->   Hand-tuned prompt  ->   Optimised prompt quality
No CoT            ->   Still no CoT       ->   CoT when it helps
No demos          ->   No demos           ->   Bootstrapped demos
top_k=3           ->   top_k=5            ->   Optimal top_k found

Low accuracy           Better accuracy         Best accuracy
Hours of tuning: 0     Hours of tuning: 4+     Hours of tuning: 0
                                               (compute does the work)
```

### Key Takeaways

| Insight | Why It Matters |
|---|---|
| Separate retrieval and generation metrics | Fix the right component |
| End-to-end optimisation beats component-level | Components interact |
| DSPy signatures replace fragile prompts | Transferable across models |
| BootstrapFewShot finds good demonstrations | Better than hand-picking |
| Chain-of-thought is not always worth the cost | Use it selectively |
| Optimisation landscape reveals insights | What matters most in YOUR data |

### When to Use Each Approach

| Scenario | Recommendation |
|---|---|
| Quick prototype | Naive RAG is fine to start |
| Production system | DSPy-style optimisation |
| Switching LLMs | Re-compile DSPy program (same code) |
| Debugging quality | Always diagnose retrieval vs generation first |

---
*Educational notebook -- NLP / RAG Optimisation Portfolio.*